In [ ]:
!pip install transformers torch datasets

In [3]:
from datasets import load_dataset

dataset = load_dataset("ingeniumacademy/reuters_articles")
dataset


README.md:   0%|          | 0.00/698 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.13M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/827k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17262 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2158 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2158 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'body'],
        num_rows: 17262
    })
    validation: Dataset({
        features: ['title', 'body'],
        num_rows: 2158
    })
    test: Dataset({
        features: ['title', 'body'],
        num_rows: 2158
    })
})

# Create New Column

In [5]:
# Create a new column: Full Article
def create_full_article_col(example):
  return {"full_article": f"TITLE:{example["title"]}\n\nBODY:{example["body"]}"}

# Mapping each record to function
dataset = dataset.map(create_full_article_col)
dataset

Map:   0%|          | 0/17262 [00:00<?, ? examples/s]

Map:   0%|          | 0/2158 [00:00<?, ? examples/s]

Map:   0%|          | 0/2158 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'body', 'full_article'],
        num_rows: 17262
    })
    validation: Dataset({
        features: ['title', 'body', 'full_article'],
        num_rows: 2158
    })
    test: Dataset({
        features: ['title', 'body', 'full_article'],
        num_rows: 2158
    })
})

In [6]:
dataset["train"][0]["full_article"]

'TITLE:BAHIA COCOA REVIEW\n\nBODY:Showers continued throughout the week in\nthe Bahia cocoa zone, alleviating the drought since early\nJanuary and improving prospects for the coming temporao,\nalthough normal humidity levels have not been restored,\nComissaria Smith said in its weekly review.\n    The dry period means the temporao will be late this year.\n    Arrivals for the week ended February 22 were 155,221 bags\nof 60 kilos making a cumulative total for the season of 5.93\nmln against 5.81 at the same stage last year. Again it seems\nthat cocoa delivered earlier on consignment was included in the\narrivals figures.\n    Comissaria Smith said there is still some doubt as to how\nmuch old crop cocoa is still available as harvesting has\npractically come to an end. With total Bahia crop estimates\naround 6.4 mln bags and sales standing at almost 6.2 mln there\nare a few hundred thousand bags still in the hands of farmers,\nmiddlemen, exporters and processors.\n    There are doubts as

# Training Our Own Tokenizer  
Here we fine-tuning a pretrained tokenizer instead of creating one from scratch

In [7]:
# Create a batch dataset for training, create an iterator object for later usage when training tokenizer

training_corpus = (
    dataset["train"][i : i + 100]["full_article"]
    for i in range(0, len(dataset["train"]), 100)
)

In [ ]:
from transformers import AutoTokenizer

# Use GPT-2 tokenizer as default
old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [9]:
tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 52000) # Vocab size of 52000

In [10]:
# Use the first 200 characters in one article as example
example = dataset["test"][2]["full_article"][:200]
example

'TITLE:CHEFS <CHEF.O> COMPLETES PRIVATE SALE\n\nBODY:Chefs International\nInc said it completed a private sale of nine mln units of its\nsecurities for 20 cts per unit for a total price of 1,800,000\ndlrs.\n'

In [11]:
# Tokenize example using gpt-2 tokenizer
old_tokenizer.tokenize(example)

['TIT',
 'LE',
 ':',
 'CH',
 'E',
 'FS',
 'Ġ<',
 'CH',
 'EF',
 '.',
 'O',
 '>',
 'ĠCOMPLE',
 'T',
 'ES',
 'ĠPR',
 'IV',
 'ATE',
 'ĠS',
 'ALE',
 'Ċ',
 'Ċ',
 'B',
 'ODY',
 ':',
 'Che',
 'fs',
 'ĠInternational',
 'Ċ',
 'Inc',
 'Ġsaid',
 'Ġit',
 'Ġcompleted',
 'Ġa',
 'Ġprivate',
 'Ġsale',
 'Ġof',
 'Ġnine',
 'Ġm',
 'ln',
 'Ġunits',
 'Ġof',
 'Ġits',
 'Ċ',
 'sec',
 'urities',
 'Ġfor',
 'Ġ20',
 'Ġc',
 'ts',
 'Ġper',
 'Ġunit',
 'Ġfor',
 'Ġa',
 'Ġtotal',
 'Ġprice',
 'Ġof',
 'Ġ1',
 ',',
 '800',
 ',',
 '000',
 'Ċ',
 'dl',
 'rs',
 '.',
 'Ċ']

In [12]:
# Tokenize example using fine-tuned tokenizer
tokenizer.tokenize(example)

['TITLE',
 ':',
 'CH',
 'EF',
 'S',
 'Ġ<',
 'CH',
 'EF',
 '.',
 'O',
 '>',
 'ĠCOMPLETES',
 'ĠPRIVATE',
 'ĠSALE',
 'Ċ',
 'Ċ',
 'BODY',
 ':',
 'Che',
 'f',
 's',
 'ĠInternational',
 'Ċ',
 'Inc',
 'Ġsaid',
 'Ġit',
 'Ġcompleted',
 'Ġa',
 'Ġprivate',
 'Ġsale',
 'Ġof',
 'Ġnine',
 'Ġmln',
 'Ġunits',
 'Ġof',
 'Ġits',
 'Ċ',
 'securities',
 'Ġfor',
 'Ġ20',
 'Ġcts',
 'Ġper',
 'Ġunit',
 'Ġfor',
 'Ġa',
 'Ġtotal',
 'Ġprice',
 'Ġof',
 'Ġ1',
 ',',
 '800',
 ',',
 '000',
 'Ċ',
 'dlrs',
 '.',
 'Ċ']

In [ ]:
# Push to hugging face
tokenizer.push_to_hub("gpt2-reuters-tokenizer")